# B2-024 — Practice p10

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** core · **Time budget:** 50 minutes  
**Concepts:** scientific-ml-inverse-problems

## Task

Implement the forward field, the grid operator, and the Tikhonov solve of Session 5.

**Contract** (all `float64`):

* `forward_field(source_xy, strength, sensors_xy, height=capstone_data.SENSOR_HEIGHT)`: `source_xy` `(N, 2)`, `strength` `(N,)`, `sensors_xy` `(S, 2)`; returns `(N, S)` with entry `strength[n] / (‖source_xy[n] - sensors_xy[s]‖² + height²)`. Use broadcasting, no Python loops.
* `grid_operator(grid_xy, sensors_xy, height=capstone_data.SENSOR_HEIGHT)`: returns `A` of shape `(S, G)` with `A[i, j]` the field at sensor `i` of a unit source at grid node `j`.
* `tikhonov_solve(A, y, lam)`: returns $(A^\top A+\lambda I)^{-1}A^\top y$ computed with **one** `torch.linalg.solve` call; `y` is `(S,)` (returns `(G,)`) or `(S, N)` (returns `(G, N)`).

**Banned (zero points if used):** `torch.inverse`, `torch.linalg.inv`, `torch.linalg.pinv`, `torch.linalg.lstsq`, NumPy linear algebra.

**Probes.** Load the unit loader with the supplied cell. `ATOL = RTOL = 1e-9` unless stated.

1. `forward_field(SRC, Q, capstone_data.SENSORS_XY)` equals `EXPECTED_FIELD`.
2. `A = grid_operator(capstone_data.GRID_XY, capstone_data.SENSORS_XY)` has shape `(8, 81)`, `A[0, 0] == 16.0`, `A[3, 40] == 3.2` (within tolerance), equals `forward_field(capstone_data.GRID_XY, torch.ones(81, dtype=torch.float64), capstone_data.SENSORS_XY).T`, and `torch.linalg.matrix_rank(A.T @ A) == 8`.
3. `tikhonov_solve(A_SMALL, Y_SMALL, 0.5)` equals `EXPECTED_TIK`, and the normal-equation residual $\lVert(A^\top A+\lambda I)u-A^\top y\rVert_2\le10^{-10}$.
4. Batched agreement: for `Y = capstone_data.val_rows("inverse")[0].double().T` (shape `(8, 200)`), `tikhonov_solve(A, Y, capstone_data.TIKHONOV_LAMBDA)` has shape `(81, 200)` and its column 0 equals `tikhonov_solve(A, Y[:, 0], capstone_data.TIKHONOV_LAMBDA)`.
5. Baseline reproduction: clip the batched densities at 0, take each column's top-3 nodes, and form the density-weighted centroid and the total mass; the resulting `(200, 3)` estimates match `capstone_data.tikhonov_grid_predict(capstone_data.val_rows("inverse")[0])` with `atol = rtol = 1e-5`.
6. Wrong variant: a local copy that drops `+ lam * I` fails probe 3 (or raises). State in one sentence why the full $81\times81$ system without $\lambda$ cannot be solved reliably.

**Required answer-check assertions.** Probes 1–6.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

SRC = torch.tensor([[0.3, 0.4], [0.75, 0.2]], dtype=torch.float64)
Q = torch.tensor([1.0, 1.5], dtype=torch.float64)
EXPECTED_FIELD = torch.tensor([
    [3.2000000000, 3.8095238095, 1.4035087719, 1.7777777778, 1.0958904110, 2.1621621622, 1.9512195122, 6.1538461538],
    [2.2556390977, 9.0909090909, 9.0909090909, 6.9767441860, 1.9607843137, 1.9607843137, 1.1857707510, 2.0979020979],
], dtype=torch.float64)
A_SMALL = torch.tensor([[1.0, 0.5, 0.0, 0.2], [0.0, 1.0, 0.5, 0.1], [0.3, 0.0, 1.0, 0.4]], dtype=torch.float64)
Y_SMALL = torch.tensor([1.0, 2.0, 0.5], dtype=torch.float64)
EXPECTED_TIK = torch.tensor([0.2320595001, 1.2237955911, 0.4371942912, 0.1186540996], dtype=torch.float64)
ATOL = 1e-9
RTOL = 1e-9

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.